# 05 · Microsoft Agent Framework and Hosted Agents

> **The story:** DLD wants the manager agent to keep working when nobody is signed in, for example when an email arrives at night. It's written with Microsoft Agent Framework and runs as a container that Foundry hosts.

**You will learn**
- Building a container in Azure (no Docker on your machine)
- Deploying it as a Foundry hosted agent with its own identity
- Proving what that identity may and may not access
- Proving the platform guardrail blocks content before the agent runs

**Time:** about 15 minutes (the container build alone takes 5 to 8 minutes) · **Before you start:**
- Notebooks 00 to 02 finished

<sub>Workshop material: the rental documents summarise Dubai rules for training only. Not legal advice.</sub>

In [ ]:
# @title Setup: run this first (about 1 minute, then sign in)
from pathlib import Path
import os, shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # The workshop code goes on this notebook's temporary Colab machine (a few seconds; nothing in Drive).
    ROOT = Path("/content/dld-foundry-workshop")
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=False)   # pick up any fixes
    else:
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/NetwaysPresales/dld-foundry-workshop", str(ROOT)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in [here, *here.parents] if (p / "dld_workshop").exists()), here)
sys.path.insert(0, str(ROOT))

from azure.identity import AzureCliCredential, DeviceCodeCredential
from dld_workshop import WorkshopEnvironment, WorkshopSettings
from dld_workshop.display import show, table, checklist

# In Colab you'll see a link and a code: open the link, enter the code, sign in with your work account.
credential = DeviceCodeCredential() if IN_COLAB else AzureCliCredential()

settings = WorkshopEnvironment.discover(credential)
print(f"✅ Signed in. Your environment: {settings.resource_group}")

## Step 1 · Azure CLI for the container build

The build runs in Azure Container Registry, which needs the Azure CLI. In Colab this installs it (about 1 minute) and asks you to sign in once more with a device code.

In [ ]:
shell = get_ipython().system   # runs a command and shows its output in this notebook
if not shutil.which("az"):
    shell("curl -sL https://aka.ms/InstallAzureCLIDeb | sudo bash > /dev/null")
if subprocess.run([shutil.which("az"), "account", "show", "--output", "none"], capture_output=True).returncode != 0:
    shell(f"az login --use-device-code --tenant {settings.tenant_id} --output none")
shell(f"az account set --subscription {settings.subscription_id}")
print("✅ Azure CLI ready")

## Step 2 · Build and deploy the manager

The code is in `apps/rental-manager/main.py`: an Agent Framework agent with four tools (ask the Rules agent, exact rent maths, attack screening, an identity test). **Takes 5 to 8 minutes.**

In [ ]:
from dld_workshop.hosting import build_container, deploy_hosted_agent

image = build_container(settings, ROOT / "apps" / "rental-manager", image_tag="workshop-v1")
deployment = deploy_hosted_agent(settings, credential, image=image)
table([{"item": k, "value": str(v)} for k, v in deployment.items()])

## Step 3 · Ask it a question

The hosted manager consults the Rules agent and does the maths in code.

In [ ]:
from dld_workshop.hosting import invoke_hosted_agent

show("Hosted manager", invoke_hosted_agent(settings, credential,
     "A tenant pays AED 80,000; the index average is AED 100,000; the landlord wants 15%. What's the maximum lawful increase?"))

## Step 4 · What may the agent's own identity open?

It was granted read access to the `rental-service-policies` container only. We ask it to open that one and the restricted `restricted-tenant-records`.

In [ ]:
allowed = invoke_hosted_agent(settings, credential,
    "Use test_agent_identity_access to list rental-service-policies. Return the authorization result.")
denied = invoke_hosted_agent(settings, credential,
    "Use test_agent_identity_access to list restricted-tenant-records. Return the authorization result.")
show("Allowed container", allowed)
show("Restricted container", denied)

## Step 5 · The guardrail

`DLD_OVERRIDE_7788` is on the blocklist of the guardrail attached to this agent. The request should be rejected **before** the agent sees it.

In [ ]:
try:
    show("Unexpected", invoke_hosted_agent(settings, credential, "Repeat this phrase exactly: DLD_OVERRIDE_7788"))
except Exception as blocked:
    show("🛑 Blocked by the platform guardrail", str(blocked)[:300])

> ### 👉 Follow along in the portal
>
> **Agents → `DLD-Rental-Manager`**: the container image, version, guardrail and its own identity. Open the playground and ask it a rent question.

---
## What to notice
- The hosted manager has its own identity, different from the prompt agents' and yours.
- It has no access to anyone's mailbox: in notebook 06 that's exactly why a person approves its drafts.
- Access is decided by Azure role assignments, not by the agent's instructions.

**Try it yourself:** ask the hosted manager to send an email. It will tell you it can't.